<a href="https://colab.research.google.com/github/Yangblingyi123/3DGS/blob/main/3DGS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [2]:
# 装CUDA 但是GCC版本有冲突，重新安装，代码失效，GCC11——torch步骤安装CUDA
# !wget https://developer.download.nvidia.com/compute/cuda/11.8.0/local_installers/cuda_11.8.0_520.61.05_linux.run
# !chmod +x cuda_11.8.0_520.61.05_linux.run
# !./cuda_11.8.0_520.61.05_linux.run --silent --toolkit --no-drm --no-man-page
# import os
# os.environ['PATH'] += ':/usr/local/cuda-11.8/bin'
# os.environ['LD_LIBRARY_PATH'] = '/usr/local/cuda-11.8/lib64:/usr/lib64-nvidia'
# !nvcc --version  # Should show CUDA 11.8

--2026-10-25 11:57:53--  https://developer.download.nvidia.com/compute/cuda/11.8.0/local_installers/cuda_11.8.0_520.61.05_linux.run
Resolving developer.download.nvidia.com (developer.download.nvidia.com)... 23.49.105.46, 23.49.105.39
Connecting to developer.download.nvidia.com (developer.download.nvidia.com)|23.49.105.46|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4336730777 (4.0G) [application/octet-stream]
Saving to: ‘cuda_11.8.0_520.61.05_linux.run’

cuda_11.8.0_520.61. 100%[===================>]   4.04G  36.1MB/s    in 91s     

2026-10-25 11:59:25 (45.5 MB/s) - ‘cuda_11.8.0_520.61.05_linux.run’ saved [4336730777/4336730777]

 Failed to verify gcc version. See log at /var/log/cuda-installer.log for details.
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Wed_Aug_20_01:58:59_PM_PDT_2025
Cuda compilation tools, release 13.0, V13.0.88
Build cuda_13.0.r13.0/compiler.36424714_0


In [3]:
# 准备GCC11 编译器
!echo "deb http://archive.ubuntu.com/ubuntu jammy main universe" | sudo tee /etc/apt/sources.list.d/jammy.list
!sudo apt-get update -qq
!sudo apt-get install -y -t jammy gcc-11 g++-11

deb http://archive.ubuntu.com/ubuntu jammy main universe
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  cpp-11 gcc-11-base libasan6 libgcc-11-dev libstdc++-11-dev libtsan0
Suggested packages:
  gcc-11-locales g++-11-multilib gcc-11-doc gcc-11-multilib libstdc++-11-doc
The following NEW packages will be installed:
  cpp-11 g++-11 gcc-11 gcc-11-base libasan6 libgcc-11-dev libstdc++-11-dev
  libtsan0
0 upgraded, 8 newly installed, 0 to remove and 48 not upgraded.
Need to get 50.6 MB of archives.
After this operation, 159 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 gcc-11-base amd64 11.2.0-19ubuntu1 [20.8 kB]
Get:2 http://archive.ubuntu.com/ubuntu 

In [7]:
# 创建一个只包含GCC11的目录
!sudo mkdir -p /opt/gcc11/bin
!sudo ln -sf /usr/bin/gcc-11 /opt/gcc11/bin/gcc
!sudo ln -sf /usr/bin/g++-11 /opt/gcc11/bin/g++
!sudo ln -sf /usr/bin/gcc-11 /opt/gcc11/bin/cc
!sudo ln -sf /usr/bin/g++-11 /opt/gcc11/bin/c++
# 用 GCC 11 作为 host 编译器安装 CUDA 11.8
!PATH=/opt/gcc11/bin:$PATH ./cuda_11.8.0_520.61.05_linux.run --silent --toolkit --toolkitpath=/usr/local/cuda-11.8 --no-drm --no-man-page; echo "exit=$?"

exit=0


In [8]:
!ls -l /usr/local/cuda-11.8/bin/nvcc
!grep -i "gcc version" /var/log/cuda-installer.log | tail -n 3

-rwxr-xr-x 1 root root 15006120 Oct 25 12:17 /usr/local/cuda-11.8/bin/nvcc
[INFO]: gcc version: gcc version 11.2.0 (Ubuntu 11.2.0-19ubuntu1) 


In [9]:
# 设置环境变量并验证 nvcc 版本
import os
os.environ['PATH'] += ':/usr/local/cuda-11.8/bin'
os.environ['LD_LIBRARY_PATH'] = '/usr/local/cuda-11.8/lib64:/usr/lib64-nvidia'
!nvcc --version  # Should show CUDA 11.8

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2022 NVIDIA Corporation
Built on Wed_Sep_21_10:33:58_PDT_2022
Cuda compilation tools, release 11.8, V11.8.89
Build cuda_11.8.r11.8/compiler.31833905_0


In [10]:
# torch 安装后必须重启会话
!pip uninstall torch torchvision torchaudio -y
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

Found existing installation: torch 2.11.0+cu130
Uninstalling torch-2.11.0+cu130:
  Successfully uninstalled torch-2.11.0+cu130
Found existing installation: torchvision 0.26.0+cu130
Uninstalling torchvision-0.26.0+cu130:
  Successfully uninstalled torchvision-0.26.0+cu130
Found existing installation: torchaudio 2.11.0+cu130
Uninstalling torchaudio-2.11.0+cu130:
  Successfully uninstalled torchaudio-2.11.0+cu130
Looking in indexes: https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 170.5 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 51.9 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 161.6 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.9/663.9 MB 16.8 MB/s  0:00:19
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 44.8 MB/s  0:00:06
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 68.8 MB/s  0:00:02
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.

In [4]:
# 查看版本
import torch
print(torch.cuda.is_available())  # Should be True
print(torch.version.cuda)        # Should be 11.8 (from PyTorch)
print(torch.cuda.get_device_name(0))  # Should show GPU

True
11.8
Tesla T4


In [2]:
!nvidia-smi
!nvcc --version

Sun Oct 25 12:27:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
import os
os.environ['PATH'] += ':/usr/local/cuda-11.8/bin'
os.environ['LD_LIBRARY_PATH'] = '/usr/local/cuda-11.8/lib64:/usr/lib64-nvidia'
!nvcc --version                 # release 11.8
import torch
print(torch.version.cuda)       # 11.8
print(torch.cuda.is_available())# True
!ls -l /opt/gcc11/bin/gcc       # 存在
!PATH=/opt/gcc11/bin:$PATH gcc --version | head -n 1

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2022 NVIDIA Corporation
Built on Wed_Sep_21_10:33:58_PDT_2022
Cuda compilation tools, release 11.8, V11.8.89
Build cuda_11.8.r11.8/compiler.31833905_0
11.8
True
lrwxrwxrwx 1 root root 15 Oct 25 12:13 /opt/gcc11/bin/gcc -> /usr/bin/gcc-11


In [7]:
# 下载GS以及安装diff-gaussian-rasterization和simple-knn
%cd /content
!git clone --recursive https://github.com/camenduru/gaussian-splatting
!pip install -q plyfile
%cd /content/gaussian-splatting
# 补齐 --no-build-isolation 需要的构建依赖
!pip install -q setuptools wheel ninja packaging
# 编译时临时前置 GCC 11，并关闭 build isolation（只对这条命令生效）
!PATH=/opt/gcc11/bin:$PATH pip install --no-build-isolation -v /content/gaussian-splatting/submodules/diff-gaussian-rasterization
!PATH=/opt/gcc11/bin:$PATH pip install --no-build-isolation -v /content/gaussian-splatting/submodules/simple-knn

/content
fatal: destination path 'gaussian-splatting' already exists and is not an empty directory.
/content/gaussian-splatting
Using pip 26.2.2 from /usr/local/lib/python3.13/dist-packages/pip (python 3.13)
Processing ./submodules/diff-gaussian-rasterization
  Running command Preparing metadata (pyproject.toml)
  running dist_info
  creating /tmp/pip-modern-metadata-bbi8nva8/diff_gaussian_rasterization.egg-info
  writing /tmp/pip-modern-metadata-bbi8nva8/diff_gaussian_rasterization.egg-info/PKG-INFO
  writing dependency_links to /tmp/pip-modern-metadata-bbi8nva8/diff_gaussian_rasterization.egg-info/dependency_links.txt
  writing top-level names to /tmp/pip-modern-metadata-bbi8nva8/diff_gaussian_rasterization.egg-info/top_level.txt
  writing manifest file '/tmp/pip-modern-metadata-bbi8nva8/diff_gaussian_rasterization.egg-info/SOURCES.txt'
  reading manifest file '/tmp/pip-modern-metadata-bbi8nva8/diff_gaussian_rasterization.egg-info/SOURCES.txt'
  writing manifest file '/tmp/pip-modern

In [8]:
# 下载示例数据集
!wget https://huggingface.co/camenduru/gaussian-splatting/resolve/main/tandt_db.zip
!unzip tandt_db.zip

--2026-10-25 12:44:44--  https://huggingface.co/camenduru/gaussian-splatting/resolve/main/tandt_db.zip
Resolving huggingface.co (huggingface.co)... 13.35.202.97, 13.35.202.40, 13.35.202.121, ...
Connecting to huggingface.co (huggingface.co)|13.35.202.97|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/64d6e7a08767727dffcfeaf6/ba454ad309f1dcb626c897350c3eeea1efdd889a9614f82e72a0be2a03a2747f?xip=2DfSS2xUyY4&user_id=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27tandt_db.zip%3B+filename%3D%22tandt_db.zip%22%3B&X-Xet-Cas-Uid=public&response-content-type=application%2Fzip&Expires=1792935884&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjRkNmU3YTA4NzY3NzI3ZGZmY2ZlYWY2L2JhNDU0YWQzMDlmMWRjYjYyNmM4OTczNTBjM2VlZWExZWZkZDg4OWE5NjE0ZjgyZTcyYTBiZTJhMDNhMjc0N2ZcXD94aXA9MkRmU1MyeFV5WTQmdXNlcl9pZD1wdWJsaWMmcmVzcG9uc2UtY29udGVudC1kaXNwb3NpdGlvbj0qJlgtWGV0LUNhcy1Va

In [ ]:
# 运行
!python train.py -s /content/gaussian-splatting/tandt/train